# Transcribe Dübendorf Gemeinderat meetings on a free Colab GPU

1. Upload the whole `duebendorf-gr-rag` folder to Google Drive under **MyDrive/duebendorf-gr-rag**
   (including `data/meetings.json` from `gr-rag scrape`, and the audio files in `data/audio/` if you downloaded them elsewhere).
2. *Runtime → Change runtime type → T4 GPU*.
3. Run all cells. Transcripts are written straight to Drive, and an interrupted run resumes where it stopped.

A T4 transcribes roughly 20–40× faster than realtime with this model, so about 40 h of meetings takes 1–2 h.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/duebendorf-gr-rag
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip -q install -e ".[transcribe,convert]"
import os; os.environ["GR_RAG_ROOT"] = os.getcwd()

### Convert the model to CTranslate2 (one-off)
The result is saved to `models/` on Drive. You can copy that folder to another machine to transcribe there with faster-whisper, without installing torch.

In [ ]:
!gr-rag convert-model

### Audio
YouTube sometimes blocks downloads from Colab's data-centre IPs ("Sign in to confirm you're not a bot").
If that happens, run `gr-rag download-audio` on your own computer (it is fast: ~30 MB per hour of meeting) and upload `data/audio/` to Drive.

In [ ]:
!gr-rag download-audio

In [ ]:
# GPU settings: float16 on CUDA is chosen automatically (compute_type: auto)
!gr-rag transcribe

In [ ]:
!gr-rag status

Done. Download (or sync) `data/transcripts/` back to your computer, then run there:
```
gr-rag protokolle
gr-rag build-index
streamlit run app.py
```